[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/26_Semantic_Layer_Basics.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 语义层基础（Semantic Layer Basics）：把知识图谱、本体和映射拼到一起

## 概览

本课把三件你已经认识的东西——知识图谱、本体、RDF 导出——拼成一个最小的*语义层*：一张类型、关系和属性都**显式映射**到本体术语的图谱，产出的 RDF 就能用 SPARQL 按共享词汇表查询。

**文档**：[API 参考](https://semantica.readthedocs.io/concepts/)

### 🎯 学习目标

- 用 `GraphBuilder` 建一张小图谱
- 用 `OntologyGenerator` 从图生成入门本体
- 写**显式的**实体类型、关系类型、属性到本体术语的映射
- 产出本体对齐的 RDF，用 `TripletStore` 存储
- 用一条小小的 SPARQL 回答一个业务问题

### 📚 前置

- [07_Building_Knowledge_Graphs.ipynb](./07_Building_Knowledge_Graphs.ipynb) —— 从实体和关系建图
- [14_Ontology.ipynb](./14_Ontology.ipynb) —— 本体生成
- [20_Triplet_Store.ipynb](./20_Triplet_Store.ipynb) —— 三元组库后端

### 先看主线

introduction 系列的收官篇：前面学的三大件在这里合体——图（07/08 篇）、本体（14 篇）、三元组与 RDF（06/20 篇）。延续 14 篇的比方：图谱是楼，本体是法；**映射就是房产登记**——把每栋楼登记到法规条目上。楼说"方言"（`works_for`），法说"官话"（`worksFor` 的 URI），登记簿逐条对照，整张图翻译成官话后才能跟全世界（SPARQL/语义网）对话。六步四段：

**备件（第 1–2 步）—— 图与法**

1. 第 1 步 建图：显式的实体与关系
2. 第 2 步 立法：`OntologyGenerator` 生成入门本体

**立契约（第 3 步）—— 映射登记**

3. 第 3 步 写映射：图的每个类型/关系/属性登记到哪个本体术语——显式对照，不做名字猜测

**翻译落库（第 4–5 步）—— 对齐与物化**

4. 第 4 步 应用映射：整张图改写成本体术语的对齐图
5. 第 5 步 落库导出：`TripletStore` 物化声明与实例，序列化成 Turtle

**一问（第 6 步）—— SPARQL**

6. 第 6 步 查询：用共享词汇表回答"谁在 Tech Corp 干什么"

> [!NOTE]
> **教学映射 vs 治理映射。** 本课的映射是演示：放在 Python dict 里，从生成的本体推导。生产级语义层要用受治理的标识符、手工设计的本体、显式的来源映射、校验（SHACL）、溯源和版本管理——那套流程见 [Advanced: Manual Ontology + Snowflake Mapping](../advanced/13_Manual_Ontology_Snowflake_Mapping.ipynb)。

## 安装

三元组库这步用内嵌的 Oxigraph 后端，装带这个 extra 的版本。至少钉 0.6.7：更早的版本可能生成没有 URI 的本体类（#1103），会让下面的映射静默失效而不是响亮报错。

```bash
pip install "semantica[tripletstore-oxigraph]>=0.6.7"
```

---

## 第 1 步：构建知识图谱

从一小组显式的实体和关系开始——两个人、一个组织、一个项目。

In [ ]:
!pip install "semantica[tripletstore-oxigraph]>=0.6.7"


In [ ]:
from semantica.kg import GraphBuilder

entities = [
    {"id": "e1", "type": "Person", "name": "Alice", "properties": {"age": 30, "role": "Engineer"}},
    {"id": "e2", "type": "Person", "name": "Bob", "properties": {"age": 35, "role": "Manager"}},
    {"id": "e3", "type": "Organization", "name": "Tech Corp", "properties": {"founded": 2010}},
    {"id": "e4", "type": "Project", "name": "Project Alpha", "properties": {"status": "active"}},
]

relationships = [
    {"source": "e1", "target": "e2", "type": "reports_to", "properties": {}},
    {"source": "e1", "target": "e3", "type": "works_for", "properties": {}},
    {"source": "e2", "target": "e3", "type": "works_for", "properties": {}},
    {"source": "e1", "target": "e4", "type": "works_on", "properties": {}},
]

builder = GraphBuilder()
knowledge_graph = builder.build({"entities": entities, "relationships": relationships})

id_to_name = {entity["id"]: entity["name"] for entity in entities}

print(f"Entities ({len(knowledge_graph['entities'])}):")
for entity in knowledge_graph["entities"]:
    print(f"  {entity['id']}: {entity['name']} ({entity['type']}) {entity['properties']}")

print(f"\nRelationships ({len(knowledge_graph['relationships'])}):")
for relationship in knowledge_graph["relationships"]:
    print(f"  {id_to_name[relationship['source']]} "
          f"--{relationship['type']}--> {id_to_name[relationship['target']]}")

## 第 2 步：生成入门本体

`OntologyGenerator` 从图记录推断 OWL 类和属性。`GraphBuilder` 把业务属性放在每个实体的 `properties` 字典里，而本体推断读的是记录字段——所以先做一个扁平的**推理视图**（知识图谱本身不动）。两个设置很关键：

- `base_uri` 把每个生成的术语放进*你的*命名空间
- `min_occurrences=1` 保留只出现一次的类（默认值 2 会把这个迷你演示图里的 `Organization` 和 `Project` 丢掉）

注意生成器会规范化名称：关系类型 `works_for` 变成本体属性 `worksFor`。这正是下一步要**显式**映射、而不是按名字匹配的原因。

In [ ]:
from semantica.ontology import OntologyGenerator

BASE_URI = "https://example.org/company/"

# Adapt the property-graph representation to the record shape consumed by
# OntologyGenerator, so age/role/founded/status become declared properties.
ontology_input = {
    "entities": [
        {
            **{key: value for key, value in entity.items() if key != "properties"},
            **entity.get("properties", {}),
        }
        for entity in knowledge_graph["entities"]
    ],
    "relationships": knowledge_graph["relationships"],
}

generator = OntologyGenerator(base_uri=BASE_URI, min_occurrences=1)
ontology = generator.generate_from_graph(ontology_input)

# OntologyGenerator calls datatype properties `data`; TripletStore's public
# ontology contract calls them `datatype`. Normalize that boundary explicitly.
store_ontology = {
    **ontology,
    "properties": [
        {**prop, "type": "datatype" if prop["type"] == "data" else prop["type"]}
        for prop in ontology["properties"]
    ],
}

print("Classes:")
for ontology_class in ontology["classes"]:
    print(f"  {ontology_class['name']:<14} {ontology_class['uri']}")

print("\nProperties:")
for prop in ontology["properties"]:
    print(f"  {prop['name']:<14} {prop['type']:<7} {prop['uri']}  "
          f"(domain={prop['domain']}, range={prop['range']})")

assert len(ontology["classes"]) == 3

## 第 3 步：把图映射到本体术语

语义层的心脏是映射契约：源的类型、关系、属性分别对应哪个本体术语。

- **实体类型**和**关系类型**：每个生成的类/属性都记录了它从哪个源名推断而来（`metadata["inferred_from"]`），映射直接从本体上读——不需要 `works_for` 与 `worksFor` 之间脆弱的名字匹配
- **属性**：扁平推理视图让 `name`、`age`、`role`、`founded`、`status` 成为真正生成的数据类型属性。每条映射因此都指向本体里已声明的术语——没有任何 URI 是到映射时才凭空发明的

In [ ]:
entity_type_mappings = {
    ontology_class["metadata"]["inferred_from"]: ontology_class["uri"]
    for ontology_class in ontology["classes"]
}

relationship_type_mappings = {
    prop["metadata"]["inferred_from"]: prop["uri"]
    for prop in ontology["properties"]
    if prop["type"] == "object"
}

datatype_property_uris = {
    prop["metadata"]["inferred_from"]: prop["uri"]
    for prop in ontology["properties"]
    if prop["type"] != "object"
}

property_mappings = datatype_property_uris

semantic_layer = {
    "graph": knowledge_graph,
    "ontology": ontology,
    "mappings": {
        "entity_type_mappings": entity_type_mappings,
        "relationship_type_mappings": relationship_type_mappings,
        "property_mappings": property_mappings,
    },
}

for mapping_name, mapping in semantic_layer["mappings"].items():
    print(f"{mapping_name}:")
    for source, target in mapping.items():
        print(f"  {source:<12} -> {target}")

# Every type and relationship in the graph must have an ontology term
assert set(entity_type_mappings) == {entity["type"] for entity in entities}
assert set(relationship_type_mappings) == {rel["type"] for rel in relationships}
assert set(property_mappings) == {"name", "age", "role", "founded", "status"}
assert set(property_mappings.values()) <= {prop["uri"] for prop in ontology["properties"]}

## 第 4 步：应用映射

应用语义层 = 改写整张图：每个类型、关系、属性键都换成本体术语。被导出和存储的是这张*对齐*图——不是原图。

In [ ]:
aligned_graph = {
    "entities": [
        {
            **entity,
            "type": entity_type_mappings[entity["type"]],
            "properties": {
                property_mappings["name"]: entity["name"],
                **{
                    property_mappings[key]: value
                    for key, value in entity["properties"].items()
                },
            },
        }
        for entity in knowledge_graph["entities"]
    ],
    "relationships": [
        {**rel, "type": relationship_type_mappings[rel["type"]]}
        for rel in knowledge_graph["relationships"]
    ],
}

print("Aligned entity sample:")
sample = aligned_graph["entities"][0]
print(f"  id:   {sample['id']}")
print(f"  type: {sample['type']}")
for key, value in sample["properties"].items():
    print(f"  {key} = {value}")

print("\nAligned relationship sample:")
print(f"  {aligned_graph['relationships'][0]['type']}")

## 第 5 步：存储并导出完整的本体对齐 RDF

`TripletStore.store()` 同时物化本体声明和对齐后的实例图。然后经 store 的公开 API 读回三元组，把完整的 RDF 图序列化成 Turtle。绕开 `RDFExporter` 的紧凑实体投影——它不带实体 `properties` 字典里的任意条目。

In [ ]:
from rdflib import Graph, Literal, URIRef
from rdflib.namespace import OWL, RDF
from semantica.triplet_store import TripletStore

store = TripletStore(backend="oxigraph")
result = store.store(aligned_graph, store_ontology)
print(f"Stored triples: {result['processed']} (failed: {result['failed']})")

rdf_graph = Graph()
for triplet in store.get_triplets():
    datatype = triplet.metadata.get("datatype")
    if datatype:
        object_term = Literal(triplet.object, datatype=URIRef(datatype))
    elif triplet.object.startswith(("http://", "https://", "urn:")):
        object_term = URIRef(triplet.object)
    else:
        object_term = Literal(triplet.object)
    rdf_graph.add((URIRef(triplet.subject), URIRef(triplet.predicate), object_term))

rdf_graph.serialize(destination="semantic_layer.ttl", format="turtle")
turtle = open("semantic_layer.ttl", encoding="utf-8").read()
print(turtle[:600])

# The exported RDF contains declarations plus mapped instance facts.
declared_datatype_properties = {
    str(subject) for subject in rdf_graph.subjects(RDF.type, OWL.DatatypeProperty)
}
assert result["failed"] == 0
assert set(property_mappings.values()) <= declared_datatype_properties
assert (
    URIRef(BASE_URI + "e1"),
    URIRef(property_mappings["role"]),
    Literal("Engineer"),
) in rdf_graph
assert (
    URIRef(BASE_URI + "e1"),
    URIRef(relationship_type_mappings["works_for"]),
    URIRef(BASE_URI + "e3"),
) in rdf_graph
print("... exported semantic_layer.ttl")

## 第 6 步：查询语义层

内嵌的 Oxigraph 后端在内存里跑，装好 `tripletstore-oxigraph` extra 即可，无需启动任何服务。组织被它映射后的 `name` 谓语约束住——这条查询问的是 *Tech Corp* 本身，不会误匹配所有组织的员工。

In [ ]:
query = f"""
SELECT ?name ?role WHERE {{
    ?person <{BASE_URI}worksFor> ?org .
    ?org <{BASE_URI}name> "Tech Corp" .
    ?person <{BASE_URI}name> ?name .
    ?person <{BASE_URI}role> ?role .
}}
ORDER BY ?name
"""
query_result = store.execute_query(query)

print("\nWho works for Tech Corp, and in which role?")
for binding in query_result.bindings:
    print(f"  {binding['name']['value']} — {binding['role']['value']}")

assert [(row["name"]["value"], row["role"]["value"]) for row in query_result.bindings] == [
    ("Alice", "Engineer"),
    ("Bob", "Manager"),
]

## 🧹 可选：清理


In [ ]:
from pathlib import Path

ttl_file = Path("semantic_layer.ttl")
if ttl_file.exists():
    ttl_file.unlink()
    print(f"Removed {ttl_file}")

## 小结

### 回顾主线

收官篇把系列三大件拼成了一个最小语义层：**备件**（楼与法：图谱 + 入门本体）、**立契约**（显式映射登记——方言到官话的对照簿）、**翻译落库**（对齐图物化成 RDF、存进三元组库）、**一问**（SPARQL 用共享词汇表回答业务问题）。至此 introduction 系列 26 篇走完：从摄取、解析、抽取，到建图、存储、分析、可视化，再到质量（冲突、去重）、治理（溯源、变更）、推理，最后在语义层合体。

一个最小的语义层是一组组合，你已经亲手建过每一件：

1. **知识图谱** —— `GraphBuilder` 从显式实体与关系构建
2. **本体** —— `OntologyGenerator` 带上你的 `base_uri`
3. **显式映射** —— 实体类型、关系类型、属性，各挂一个本体术语
4. **本体对齐的 RDF** —— 映射应用到图上，`TripletStore` 物化，从 store 自己的三元组序列化成 Turtle
5. **可查询的库** —— `TripletStore`（内嵌 Oxigraph）用 SPARQL 在共享词汇表上答题

### 下一步去哪

这套流程的生产版——手工设计的治理本体、来自数据仓库的显式映射、n 元建模、SHACL 校验、溯源与版本管理——见 [Advanced: Manual Ontology + Snowflake Mapping](../advanced/13_Manual_Ontology_Snowflake_Mapping.ipynb)。